## Check that your code does what you think it does with tests

* Testing your code ensures that it does what you expect it to do. 

* Test your code before deploying to production.

*  Cost of an issue is high in production.
* Tests gives you the confidence to move fast and not have to worry about a code change breaking any existing feature (aka regression)
* Tests force you to think carefully about your code design.
* Without tests you can dump all your code logic into one script/function, but inorder to write good tests we need modular and DRY code.
* **Note** Code tests, check that the code does what it is supposed to do and are run before code is deployed to production. Data Quality checks are part of the pipeline and are a separate concept.|

#### Example

Consider the following data transformation function. Let's see why unit testing is important.



In [ ]:
# Create a SparkSession
from pyspark.sql import Row, SparkSession

# Create Spark session
spark = SparkSession.builder.appName("06_code_testing").master("local[*]").getOrCreate()

In [ ]:
! uv run ./silver/fct_order_lines.py --start-time '2025-01-01 00:00:00' --end-time '2026-01-01 00:00:00'

In [ ]:
import pyspark.sql.functions as F


# function to get standard metrics from local.silver.fct_order_lines
def get_standard_metrics(
    fct_order_lines_df,
    grains=[F.date_format(F.col("created_at"), "yyyy-MM-dd").alias("created_dt")],
):
    # metrics, we can add more as needed
    metrics = [
        F.sum(F.col("unit_price") * F.col("quantity") - F.col("discount_amt")).alias(
            "sum_total_amount"
        ),
        F.max(F.col("unit_price") * F.col("quantity") + F.col("discount_amt")).alias(
            "max_total_amount"
        ),
    ]

    grouped_fct_order_lines_df = fct_order_lines_df.groupBy(*grains).agg(*metrics)
    # Unpacking: 
    # if metrics = [a, b] 
    # agg(*metrics) is the same as agg(a, b) # list is removed
    return grouped_fct_order_lines_df

In [ ]:
get_standard_metrics(spark.table("local.silver.fct_order_lines")).limit(5).toPandas()

In [ ]:
import pyspark.sql.functions as F
from chispa import assert_df_equality

fct_order_lines_data = [
    ("2025-01-01T00:00:00", "order-1", "variant-A", 2, 100.0, 10.0),
    ("2025-01-01T01:00:00", "order-2", "variant-B", 1, 200.0, 0.0),
    ("2025-01-02T00:00:00", "order-3", "variant-A", 3, 150.0, 15.0),
]
fct_order_lines_df = spark.createDataFrame(
    fct_order_lines_data,
    [
        "created_at",
        "order_id",
        "variant_id",
        "quantity",
        "unit_price",
        "discount_amt",
    ],
)


# 1a. Default grain (created_dt)
result = get_standard_metrics(fct_order_lines_df)
assert "created_dt" in result.columns
assert result.count() == 2, "get_standard_metrics does not work with default grain"

# 1b. Single custom grain (created_at)
result = get_standard_metrics(fct_order_lines_df, grains=[F.col("created_at")])
assert result.count() == 3, "get_standard_metrics does not work with custom grain"

# 1c. Multiple grains (created_at + variant_id)
result = get_standard_metrics(
    fct_order_lines_df, grains=[F.col("created_at"), F.col("variant_id")]
)
assert (
    result.count() == 3
), "get_standard_metrics does not work with multiple custom grains"

# 1d. Expression grain (year-month)
result = get_standard_metrics(
    fct_order_lines_df,
    grains=[F.date_format(F.col("created_at"), "yyyy-MM").alias("created_month")],
)
assert "created_month" in result.columns
assert (
    result.count() == 1
), "get_standard_metrics does not work with expressions as grains"


expected_df = spark.createDataFrame(
    [
        ("2025-01-01", 390.0, 200.0),
        ("2025-01-02", 435.0, 435.0),
    ],
    ["created_dt", "sum_total_amount", "max_total_amount"],
)

assert_df_equality(
    get_standard_metrics(fct_order_lines_df), expected_df, ignore_row_order=True
)

print("✅ All tests passed")

In [ ]:
import pyspark.sql.functions as F


# function to get standard metrics from local.silver.fct_order_lines
def get_standard_metrics(
    fct_order_lines_df,
    grains=[F.date_format(F.col("created_at"), "yyyy-MM-dd").alias("created_dt")],
):
    # metrics, we can add more as needed
    metrics = [
        F.sum(F.col("unit_price") * F.col("quantity") - F.col("discount_amt")).alias(
            "sum_total_amount"
        ),
        F.max(F.col("unit_price") * F.col("quantity") - F.col("discount_amt")).alias(
            "max_total_amount"
        ),
    ]

    grouped_fct_order_lines_df = fct_order_lines_df.groupBy(*grains).agg(*metrics)
    return grouped_fct_order_lines_df

In [ ]:
import pyspark.sql.functions as F
from chispa import assert_df_equality

fct_order_lines_data = [
    ("2025-01-01T00:00:00", "order-1", "variant-A", 2, 100.0, 10.0),
    ("2025-01-01T01:00:00", "order-2", "variant-B", 1, 200.0, 0.0),
    ("2025-01-02T00:00:00", "order-3", "variant-A", 3, 150.0, 15.0),
]
fct_order_lines_df = spark.createDataFrame(
    fct_order_lines_data,
    [
        "created_at",
        "order_id",
        "variant_id",
        "quantity",
        "unit_price",
        "discount_amt",
    ],
)


# 1a. Default grain (created_dt)
result = get_standard_metrics(fct_order_lines_df)
assert "created_dt" in result.columns
assert result.count() == 2, "get_standard_metrics does not work with default grain"

# 1b. Single custom grain (created_at)
result = get_standard_metrics(fct_order_lines_df, grains=[F.col("created_at")])
assert result.count() == 3, "get_standard_metrics does not work with custom grain"

# 1c. Multiple grains (created_at + variant_id)
result = get_standard_metrics(
    fct_order_lines_df, grains=[F.col("created_at"), F.col("variant_id")]
)
assert (
    result.count() == 3
), "get_standard_metrics does not work with multiple custom grains"

# 1d. Expression grain (year-month)
result = get_standard_metrics(
    fct_order_lines_df,
    grains=[F.date_format(F.col("created_at"), "yyyy-MM").alias("created_month")],
)
assert "created_month" in result.columns
assert (
    result.count() == 1
), "get_standard_metrics does not work with expressions as grains"


expected_df = spark.createDataFrame(
    [
        ("2025-01-01", 390.0, 200.0),
        ("2025-01-02", 435.0, 435.0),
    ],
    ["created_dt", "sum_total_amount", "max_total_amount"],
)

assert_df_equality(
    get_standard_metrics(fct_order_lines_df), expected_df, ignore_row_order=True
)

print("✅ All tests passed")

* Notice how a small mistype can create incorrect data

* Tests help us catch these before impacting production
* We can now be confident when we use the `get_standard_metrics` function since we have ensured that it works as expected.
* Fixture: The fake data `fct_order_lines_df` we create just to test our code
* Creating data fixtures are extremely manual and time consuming for complex pipelines (joins/filters/group/unions) with multiple inputs.
* In SWE `unit` tests refer to code that test a *unit of code*

*  Depending on how “modular” your pipeline code is, the concept of a unit is malleable

* It is up to us, the data engineers to ensure that we test the right things.

#### Exercise [15 min]

* Assume that you are testing the transform function of `silver/dim_customer.py` 
1. Create the input fixtures
2. Check that the output of the transform function is what you expect it to be

**Hint**: Use `spark.createDataFrame` to create fixtures, create address as
```python
[Row(is_default=True, address="789 Pine Rd, Ogdenville, NY, US"),],
```

In [ ]:
from chispa import assert_df_equality
assert_df_equality?

In [ ]:


from chispa import assert_df_equality
from pyspark.sql import functions as F
from silver.dim_customer import transform

# ── fixtures ─────────────────────────────────────────────────────────────────

customer_df = spark.createDataFrame(
    [
        (
            "cust-1",
            "alice@example.com",
            "Alice Smith",
            "555-0001",
            "active",
            "2025-01-01T00:00:00",
            "2025-01-01T00:00:00",
        ),
        (
            "cust-2",
            "bob@example.com",
            "Bob Jones",
            "555-0002",
            "active",
            "2025-01-02T00:00:00",
            "2025-01-02T00:00:00",
        ),
        (
            "cust-3",
            "eve@example.com",
            "Eve Davis",
            "555-0003",
            "inactive",
            "2025-01-03T00:00:00",
            "2025-01-03T00:00:00",
        ),
    ],
    [
        "customer_id",
        "email",
        "full_name",
        "phone",
        "status",
        "created_at",
        "updated_at",
    ],
)

customer_address_df = spark.createDataFrame(
    [
        ("cust-1", True, "123 Main St", "Springfield", "IL", "US"),
        ("cust-1", False, "456 Oak Ave", "Shelbyville", "IL", "US"),
        ("cust-2", True, "789 Pine Rd", "Ogdenville", "NY", "US"),
    ],
    ["customer_id", "is_default", "line1", "city", "state", "country"],
)

expected_data = [
    (
        "cust-1",
        "alice@example.com",
        "Alice Smith",
        "555-0001",
        "active",
        "2025-01-01T00:00:00",
        "2025-01-01T00:00:00",
        [
            Row(is_default=False, address="456 Oak Ave, Shelbyville, IL, US"),
            Row(is_default=True, address="123 Main St, Springfield, IL, US"),
        ],
    ),
    (
        "cust-2",
        "bob@example.com",
        "Bob Jones",
        "555-0002",
        "active",
        "2025-01-02T00:00:00",
        "2025-01-02T00:00:00",
        [
            Row(is_default=True, address="789 Pine Rd, Ogdenville, NY, US"),
        ],
    ),
    (
        "cust-3",
        "eve@example.com",
        "Eve Davis",
        "555-0003",
        "inactive",
        "2025-01-03T00:00:00",
        "2025-01-03T00:00:00",
        [Row(is_default=None, address=None)],
    ),
]
expected_df = spark.createDataFrame(
    expected_data,
    [
        "customer_id",
        "email",
        "full_name",
        "phone",
        "status",
        "created_at",
        "updated_at",
        "addresses",
    ],
)

transformed_df = transform(
    {"customer_df": customer_df, "customer_address_df": customer_address_df}, spark
)

assert_df_equality(
    transformed_df,
    expected_df,
    ignore_row_order=True,
    ignore_nullable=True,
)

print("✅ Test passed")

* Creating fixtures for data pipelines is tedious, especially with complex joins and filters. Sometimes, writing tests is not worth the effort.
* I find LLMs to be extremely effective in generating the query to access production data given the function to be tested.
* When it comes to data pipeline the key here is to know what to test for
* Tips: Test output schema and logic of transformed columns; skip untransformed columns.

## Use Pytest to manage tests

* In complex projects you’d want all your tests to run with a simple command and have a system to manage your fixtures so you are not having to create and delete them with custom code
* This is where pytest helps
* pytest is a python module to help run test code and makes writing tests more ergonomical
* it works by looking for python files with a specific name: test_*
* Within each of those python files, it looks for functions called test_* or class called Test*

#### Example

* Let's look at how to use Pytest to test `silver/dim_customer.py` script.
* In this script we test the `transform` function multiple times independently.

[test_silver_dim_customer.py](./test_silver_dim_customer.py)

In [ ]:
! uv run pytest ./test_silver_dim_customer.py

* Pytest enables you to do the following.
  1. Define and re-use fixtures with the @fixture decorator.
  2. Change your code behaviour during testing (aka Patching)
  3. Automatically know which classes/functions are tests based on their naming pattern.
* The testing process has four steps:
  1. The **arrange** step involves setting up fixtures, test tables, and other resources. We set the scene for our code to run.
  2. The **act** step involves invoking the code we want to test.
  3. The **assert** step checks that the output of our code (from the act step) matches our expectations.
  4. The **cleanup** step involves removing any data/db tables, etc, that remain from the arranging step.

#### Exercise [15 min]

Create a `test_silver_fct_order_lines.py` similar to `test_silver_dim_customer.py` to test the transform function in ./silver/fct_order_lines.py. Test these 2 operations

* is_discounted is correctly derived
* effective_unit_price is correctly calculated

In [ ]:
! uv run pytest ./test_silver_fct_order_lines.py

[test_silver_fct_order_lines](test_silver_fct_order_lines.py)

In [ ]:
! uv run pytest ./

* The sparksession and fixture are recreated in each file.
* Pytest enables you to define fixtures in one file called conftest.py
* In our example conftest.py, we create a shareable

  - SparkSession 

  - Dataframe for customer, customer_address & order_lines

* The fixtures in [conftest.py](tests/conftest.py) can be accessed by passing them as input parameters to the test function.

#### Exercise [15 min]

For this exercise, do the following:

1. Create a tests folder.
2. Create a conftest.py file under /tests/conftest.py and include a SparkSession, and all the fixtures we have defined.
3. The fixtures should be defined in conftest.py, and the test files should only have the necessary fixtures as input arguments

In [ ]:
! uv run pytest ./tests


* When setting up fixtures for testing, we can create them at different scopes. The different scopes are
    1. **Session level**: The setup is run once before we start testing all the test files, and the teardown is run once after all the testing is done.
    2. **Class level**: The setup and teardown are before and after each Class (usually named class Testxyz).
    3. **Function level**: The setup and teardown are before and after each function (usually named test_).
* We use the scope="session" to define the scope for our fixtures
* We can also setup other infrastructure using the `setup-teardown` pattern. In `conftest.py` for the sparksession we use yield to do this.
```python
@pytest.fixture(scope="session")
def spark():
    spark = (
        SparkSession.builder
        .master("local")
        .appName("test_ecommerce")
        .getOrCreate()
    )
    yield spark
    spark.stop()
```

![Conftest Flow](images/conftest-flow.png)

* Everything before yield is setup, everything after is teardown. Pytest will call `spark.stop()` automatically once all tests in the session have finished.
* If you are creating fake tables or any persistent fixture (e.g. cloud data, etc) make sure to stop/delete/teardown after the tests are complete

## Ensure systems work together as expected with Integration tests

* data pipelines usually involve multiple interacting systems, integration tests are meant to test these interactions
* Manually checking may suffice for a simple pipeline. But as the pipeline, inputs, and data team grow in size, it will become increasingly difficult to check every system interaction
* Integration tests usually involve multiple systems simulated with with containers/mocks

#### Example

Let’s write integration tests to check 
1. `overwritePartitions` work as expected

In [ ]:
! uv run pytest ./test_fct_order_lines_integration.py

[test_fct_order_lines_integration.py](test_fct_order_lines_integration.py) to test [fct_order_lines.py](silver/fct_order_lines.py)

* In this above integration test, we introduce

  - Monkeypatching, which refers to updating code to be tested. In our case, we update the hardcoded table name with the test table name

* Do not test integration features; test that you are using them correctly. 

* For example, do not test whether a create table destination creates a destination table; instead, run your pipeline and check that the destination table produces the expected output with the expected schema.

* Do not test functionality that doesn't matter. e.g., if you are just dumping data to an S3 output path and don’t care about the schema, just test that the data is written to the right path.

* If your external systems are flaky (random failures, changing schemas, etc.), do not write integration tests; instead, work on setting up a data contract with the data provider.

* Do not test framework features (e.g., Airflow retry), but test that you have used it as expected (e.g., Task branching logic).

## Recap

In this section, we saw why code testing is important. We covered
1. Writing tests to ensure that our code does what we expect it to
2. How to use Pytests to manage tests
3. Why integration tests are important